In [0]:
import os, sys, uuid, yaml
REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from pyspark.sql import functions as F
from src.quality.quarantine import split, to_rejects, write_rejects, reconcile, REJECTS
from src.transforms.vehicle_positions import VP_KEY

with open(f"{REPO_ROOT}/config/quality_rules.yaml") as fh:
    CONFIG_DOC = yaml.safe_load(fh)

def quarantine_rules(table):
    entry = next((e for e in CONFIG_DOC if e["table"] == table), None)
    return [r for r in (entry or {}).get("rules", []) if r.get("quarantine")]

spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {REJECTS} (
    run_id STRING, first_rejected_at TIMESTAMP, last_rejected_at TIMESTAMP,
    source_table STRING, layer STRING, natural_key STRING,
    reject_reasons ARRAY<STRING>, reject_reason STRING, row_json STRING)
""")

VP = "transit.silver.vehicle_positions"
rules = quarantine_rules(VP)
print(f"{len(rules)} quarantine rules:", [r["name"] for r in rules])

In [0]:
vp = spark.table(VP)
valid, rejected = split(vp, rules)

n_in, n_valid, n_rej = vp.count(), valid.count(), rejected.count()
print(f"input {n_in:,} · valid {n_valid:,} · rejected {n_rej:,}")
reconcile(n_in, n_valid, n_rej, VP)

(rejected.select(F.explode("_reject_reasons").alias("reason"))
   .groupBy("reason").count().orderBy(F.desc("count")).display())

rejected.select("vehicle_id", "route_id", "latitude", "longitude",
                "occupancy_pct", "_reject_reasons").limit(20).display()

In [0]:
run_id = str(uuid.uuid4())[:8]

write_rejects(spark, to_rejects(rejected, VP, "silver", VP_KEY, run_id))

(valid.write.format("delta").mode("overwrite")
   .option("overwriteSchema", "true").saveAsTable(VP))

after = spark.table(VP).count()
n_rejects = spark.table(REJECTS).filter(F.col("source_table") == VP).count()
print(f"silver.vehicle_positions {n_in:,} -> {after:,}")
print(f"{REJECTS}: {n_rejects:,} rows for this table")
reconcile(n_in, after, n_rejects, "after write")

In [0]:
rj = spark.table(REJECTS)

(rj.groupBy("source_table", "reject_reason")
   .agg(F.count("*").alias("rows"),
        F.min("first_rejected_at").alias("first_seen_bad"),
        F.max("last_rejected_at").alias("last_seen_bad"))
   .orderBy(F.desc("rows")).display())

sample = rj.first()
print(sample.natural_key)
print(sample.row_json[:400])

The above cell fails as silver is already clean !!


In [0]:
# reprocessing path: re-evaluate rejects against the current rules and
# return anything that now passes. Rules change; rejects should be revisited.
from pyspark.sql.functions import from_json

# the schema of the rows that were rejected, taken from the table they came from
vp_schema = spark.table(VP).schema

restored = (rj.filter(f"source_table = '{VP}'")
              .select(from_json("row_json", vp_schema).alias("r"))
              .select("r.*"))

still_valid, still_bad = split(restored, rules)
print(f"rejects re-evaluated: {still_valid.count()} now pass · {still_bad.count()} still fail")